# Train ICSRec on Amazon Grocery and Gourmet Food (15-core)

**Course:** CCAI 422 Recommender Systems — University of Jeddah, Spring 2025/2026

**Paper:** Qin et al., *Intent Contrastive Learning with Cross Subsequences for Sequential Recommendation*, WSDM 2024.

**Goal:** Run a baseline training of ICSRec on our preprocessed Grocery dataset, capture metrics for the report, and save the trained checkpoint to Google Drive.

### Prerequisites
- Runtime → Change runtime type → any GPU
- The preprocessing notebook has been run, so `/content/drive/MyDrive/ICSRec_project/Grocery_and_Gourmet_Food.txt` exists

### What this notebook does
1. Verifies GPU is enabled
2. Mounts Google Drive
3. Clones the ICSRec repository
4. Installs the `faiss-gpu` dependency (used for the intent K-means clustering)
5. Copies our preprocessed Grocery file into ICSRec's `data/` folder
6. Runs baseline training
7. Backs up logs and the trained checkpoint to Drive

## Step 1: Verify GPU is enabled

ICSRec requires a GPU for two reasons: (a) the transformer training itself, and (b) the `faiss-gpu` K-means used to cluster user-intent embeddings every epoch. If this cell prints `False`, change the runtime type before proceeding.

In [1]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU is not enabled. Go to Runtime -> Change runtime type -> select a GPU.')

!nvidia-smi | head -20

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Wed May 27 10:44:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             44W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                   

## Step 2: Mount Google Drive

We need the preprocessed `Grocery_and_Gourmet_Food.txt` from the previous notebook.

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ICSRec_project'
DATA_FILE = os.path.join(DRIVE_DIR, 'Grocery_and_Gourmet_Food.txt')

assert os.path.exists(DATA_FILE), f'Could not find {DATA_FILE}. Did you run the preprocessing notebook?'
print('Preprocessed data found:', DATA_FILE)
print('Size: %.2f MB' % (os.path.getsize(DATA_FILE) / 1e6))

Mounted at /content/drive
Preprocessed data found: /content/drive/MyDrive/ICSRec_project/Grocery_and_Gourmet_Food.txt
Size: 1.68 MB


## Step 3: Install faiss-gpu

ICSRec uses Facebook's `faiss` library for the K-means intent clustering. We install the GPU-accelerated version.

*If `faiss-gpu` fails to install due to CUDA version mismatch, fall back to `faiss-cpu` and we'll patch the code in Step 5b.*

In [3]:
# Try the GPU build first. On modern Colab (CUDA 12), the conda-style faiss-gpu
# package isn't always pip-installable, so we try faiss-gpu first and fall back
# to faiss-cpu if needed.
import subprocess

def try_install(pkg):
    print(f'Trying: pip install {pkg}')
    r = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True, text=True)
    return r.returncode == 0

FAISS_BACKEND = None
for candidate in ['faiss-gpu', 'faiss-cpu']:
    if try_install(candidate):
        FAISS_BACKEND = candidate
        print(f'Installed: {candidate}')
        break

if FAISS_BACKEND is None:
    raise RuntimeError('Neither faiss-gpu nor faiss-cpu could be installed.')

# Verify
import faiss
print('faiss imported successfully, version:', faiss.__version__)

Trying: pip install faiss-gpu
Trying: pip install faiss-cpu
Installed: faiss-cpu
faiss imported successfully, version: 1.14.2


## Step 4: Clone the ICSRec repository

In [4]:
%cd /content
!rm -rf ICSRec
!git clone https://github.com/QinHsiu/ICSRec.git
%cd ICSRec
!ls

/content
Cloning into 'ICSRec'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 198 (delta 34), reused 109 (delta 18), pack-reused 55 (from 1)
Receiving objects: 100% (198/198), 28.28 MiB | 40.84 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/ICSRec
data	 performances.md  README.md	    SECURITY.md
LICENSE  pics		  requirements.txt  src


## Step 5: Copy our preprocessed data into ICSRec

ICSRec's `main.py` reads `--data_dir/--data_name.txt`. We place our file at `data/Grocery_and_Gourmet_Food.txt` and use `--data_name Grocery_and_Gourmet_Food`.

In [5]:
import shutil

target = '/content/ICSRec/data/Grocery_and_Gourmet_Food.txt'
shutil.copy(DATA_FILE, target)
print('Copied to:', target)
print('Size: %.2f MB' % (os.path.getsize(target) / 1e6))

# Sanity-check: first three lines should look like "<userid> <itemid> <itemid> ..."
with open(target) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        toks = line.split()
        print(f'user {toks[0]}: sequence length = {len(toks)-1}, first 10 items = {toks[1:11]}')

Copied to: /content/ICSRec/data/Grocery_and_Gourmet_Food.txt
Size: 1.68 MB
user 1: sequence length = 20, first 10 items = ['576', '2336', '842', '2077', '8465', '1996', '705', '5949', '1253', '848']
user 2: sequence length = 17, first 10 items = ['1685', '8397', '7700', '634', '5159', '3913', '696', '7703', '3653', '3245']
user 3: sequence length = 29, first 10 items = ['6937', '4240', '4023', '4705', '6552', '1095', '5135', '5369', '5956', '5735']


## Step 5b: Patch the code if faiss-gpu is unavailable

ICSRec hardcodes faiss-gpu calls (`StandardGpuResources`, `GpuIndexFlatL2`). If we had to fall back to `faiss-cpu`, we patch `src/models.py` to use the CPU index instead. K-means runs slower on CPU, but for our dataset size (12.7K users × 256 clusters) it's still acceptable.

In [6]:
if FAISS_BACKEND == 'faiss-cpu':
    print('Patching src/models.py to use CPU faiss...')
    path = '/content/ICSRec/src/models.py'
    with open(path) as f:
        code = f.read()
    # Replace the GPU resource setup with a CPU index
    patched = code.replace(
        '        res = faiss.StandardGpuResources()\n'
        '        res.noTempMemory()\n'
        '        cfg = faiss.GpuIndexFlatConfig()\n'
        '        cfg.useFloat16 = False\n'
        '        cfg.device = self.gpu_id\n'
        '        index = faiss.GpuIndexFlatL2(res, hidden_size, cfg)',
        '        index = faiss.IndexFlatL2(hidden_size)  # CPU fallback'
    )
    if patched == code:
        print('WARNING: patch text not found; the model file may have changed. Inspect manually.')
    else:
        with open(path, 'w') as f:
            f.write(patched)
        print('Patched.')
else:
    print('faiss-gpu is installed; no patching needed.')

Patching src/models.py to use CPU faiss...
Patched.


## Step 6: Run baseline training

We use the same hyperparameters as the paper's Sports/Beauty configurations (these worked best for the paper across similar Amazon datasets):

- `rec_weight=1.0` — recommendation loss weight
- `lambda_0=0.3` — coarse-grain intent contrastive loss weight
- `beta_0=0.1` — fine-grain contrastive loss weight
- `intent_num=256` — number of K-means clusters
- `f_neg` — enable the false-negative mitigation (FNM) component
- `epochs=200` — capped; early stopping triggers when NDCG@20 doesn't improve for 40 epochs

Training writes its log to `output/ICSRec-SAS-Grocery_and_Gourmet_Food-0.txt`.

In [8]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 92.8 MB/s eta 0:00:00


In [9]:
%cd /content/ICSRec/src

# Note: `--data_name` matches the filename we copied (without .txt).
# `--model_idx 0` is just a tag used in saved checkpoint/log filenames.
# We let main.py auto-generate the supervised training file (data/Grocery_and_Gourmet_Food_1.txt)
# on the first run via its DS(...) preprocessing call.

!python main.py \
    --data_name Grocery_and_Gourmet_Food \
    --rec_weight 1.0 \
    --lambda_0 0.3 \
    --beta_0 0.1 \
    --f_neg \
    --intent_num 256 \
    --hidden_dropout_prob 0.5 \
    --attention_probs_dropout_prob 0.5 \
    --model_idx 0 \
    --epochs 200

/content/ICSRec/src
Using Cuda: True
--------------------Configure Info:------------
data_dir                       :                            ../data/
output_dir                     :                              output
data_name                      :            Grocery_and_Gourmet_Food
encoder                        :                                 SAS
do_eval                        :                                   0
model_idx                      :                                   0
gpu_id                         :                                   0
noise_ratio                    :                                 0.0
temperature                    :                                 1.0
intent_num                     :                                 256
sim                            :                                 dot
model_name                     :                              ICSRec
hidden_size                    :                                  64
num_hidden_layers 

## Step 7: Back up logs and checkpoint to Google Drive

Colab sessions are not persistent — if the runtime disconnects we lose `/content`. Anything we want to keep must be in `/content/drive`.

In [10]:
backup_dir = os.path.join(DRIVE_DIR, 'training_outputs_baseline')
os.makedirs(backup_dir, exist_ok=True)

# copy everything the training step produced
src_output = '/content/ICSRec/src/output'
if os.path.isdir(src_output):
    for fn in os.listdir(src_output):
        sp = os.path.join(src_output, fn)
        dp = os.path.join(backup_dir, fn)
        shutil.copy(sp, dp)
        print(f'Backed up: {fn} ({os.path.getsize(dp)/1e6:.2f} MB)')

# also back up the auto-generated supervised-training data file
ds_file = '/content/ICSRec/data/Grocery_and_Gourmet_Food_1.txt'
if os.path.exists(ds_file):
    shutil.copy(ds_file, os.path.join(DRIVE_DIR, 'Grocery_and_Gourmet_Food_1.txt'))
    print('Backed up the supervised-training data file.')

print('\nBackup dir contents:')
for fn in sorted(os.listdir(backup_dir)):
    print(' ', fn)

Backed up: ICSRec-SAS-Toys_and_Games-0.pt (3.47 MB)
Backed up: ICSRec-SAS-ml-1m-0.pt (1.30 MB)
Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-0.txt (0.04 MB)
Backed up: ICSRec-SAS-Beauty-0.pt (3.52 MB)
Backed up: ICSRec-GRU-Toys_and_Games-0.pt (3.39 MB)
Backed up: ICSRec-GRU-Beauty-0.pt (3.43 MB)
Backed up: ICSRec-GRU-ml-1m-0.pt (1.21 MB)
Backed up: ICSRec-SAS-Sports_and_Outdoors-0.pt (5.12 MB)
Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-0.pt (2.67 MB)
Backed up: ICSRec-GRU-Sports_and_Outdoors-0.pt (5.03 MB)
Backed up the supervised-training data file.

Backup dir contents:
  ICSRec-GRU-Beauty-0.pt
  ICSRec-GRU-Sports_and_Outdoors-0.pt
  ICSRec-GRU-Toys_and_Games-0.pt
  ICSRec-GRU-ml-1m-0.pt
  ICSRec-SAS-Beauty-0.pt
  ICSRec-SAS-Grocery_and_Gourmet_Food-0.pt
  ICSRec-SAS-Grocery_and_Gourmet_Food-0.txt
  ICSRec-SAS-Sports_and_Outdoors-0.pt
  ICSRec-SAS-Toys_and_Games-0.pt
  ICSRec-SAS-ml-1m-0.pt


## Step 8: Read the final metrics

The last line of the log file contains the test-set metrics.

In [11]:
log_path = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-0.txt'
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print('========== Last 20 lines of training log ==========')
    for line in lines[-20:]:
        print(line.rstrip())
else:
    print('Log file not found at', log_path)
    print('Look in /content/ICSRec/src/output/ for the actual log filename.')

========== Last 20 lines of training log ==========
{'Epoch': 174, 'HIT@5': '0.0365', 'NDCG@5': '0.0236', 'HIT@10': '0.0604', 'NDCG@10': '0.0313', 'HIT@20': '0.0909', 'NDCG@20': '0.0390'}
{'epoch': 175, 'rec_avg_loss': '7.1297', 'icl_avg_loss': '2.6927', 'joint_avg_loss': '9.8224'}
{'Epoch': 175, 'HIT@5': '0.0381', 'NDCG@5': '0.0247', 'HIT@10': '0.0611', 'NDCG@10': '0.0320', 'HIT@20': '0.0907', 'NDCG@20': '0.0395'}
{'epoch': 176, 'rec_avg_loss': '7.1309', 'icl_avg_loss': '2.6923', 'joint_avg_loss': '9.8232'}
{'Epoch': 176, 'HIT@5': '0.0390', 'NDCG@5': '0.0251', 'HIT@10': '0.0606', 'NDCG@10': '0.0320', 'HIT@20': '0.0929', 'NDCG@20': '0.0401'}
{'epoch': 177, 'rec_avg_loss': '7.1301', 'icl_avg_loss': '2.6924', 'joint_avg_loss': '9.8225'}
{'Epoch': 177, 'HIT@5': '0.0386', 'NDCG@5': '0.0250', 'HIT@10': '0.0599', 'NDCG@10': '0.0318', 'HIT@20': '0.0904', 'NDCG@20': '0.0395'}
{'epoch': 178, 'rec_avg_loss': '7.1299', 'icl_avg_loss': '2.6912', 'joint_avg_loss': '9.8211'}
{'Epoch': 178, 'HIT@5': 

## Done. What's next?

**We now have a baseline.**:
- HIT@5, HIT@10, HIT@20
- NDCG@5, NDCG@10, NDCG@20

**Next steps for Phase 2:**
**Hyperparameter tuning** — try 3 settings of `intent_num` (e.g. 64, 128, 256, 512) and report the best one. Repeat the cell above with different flags.

**Save every run's log** — use a different `--model_idx` per run so the logs don't overwrite each other.

# Hyperparameter Tuning

In [12]:

# check before hyperparameter tuning runs

import os, shutil, subprocess

# drive still mounted?
if not os.path.exists('/content/drive/MyDrive'):
    print('Drive not mounted, mounting...')
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ICSRec_project'
print('Drive OK:', DRIVE_DIR)

# ICSRec repo still here?
if not os.path.exists('/content/ICSRec/src/main.py'):
    print('ICSRec missing, re-cloning...')
    subprocess.run(['rm', '-rf', '/content/ICSRec'])
    subprocess.run(['git', 'clone', 'https://github.com/QinHsiu/ICSRec.git', '/content/ICSRec'])
    # re-patch faiss
    path = '/content/ICSRec/src/models.py'
    with open(path) as f:
        code = f.read()
    patched = code.replace(
        '        res = faiss.StandardGpuResources()\n'
        '        res.noTempMemory()\n'
        '        cfg = faiss.GpuIndexFlatConfig()\n'
        '        cfg.useFloat16 = False\n'
        '        cfg.device = self.gpu_id\n'
        '        index = faiss.GpuIndexFlatL2(res, hidden_size, cfg)',
        '        index = faiss.IndexFlatL2(hidden_size)  # CPU fallback'
    )
    with open(path, 'w') as f:
        f.write(patched)
    print('Re-patched faiss to CPU')
print('ICSRec OK')

# grocery data file in place?
target = '/content/ICSRec/data/Grocery_and_Gourmet_Food.txt'
if not os.path.exists(target):
    shutil.copy(os.path.join(DRIVE_DIR, 'Grocery_and_Gourmet_Food.txt'), target)
    print('Re-copied Grocery data')
print('Data OK:', target)

# Restore the supervised training file
ds = '/content/ICSRec/data/Grocery_and_Gourmet_Food_1.txt'
src_ds = os.path.join(DRIVE_DIR, 'Grocery_and_Gourmet_Food_1.txt')
if not os.path.exists(ds) and os.path.exists(src_ds):
    shutil.copy(src_ds, ds)
    print('Restored supervised training file from Drive')
print('Supervised data OK' if os.path.exists(ds) else 'Will be regenerated on first run')

# faiss + gensim installed
try:
    import faiss
    import gensim
    print('Packages OK')
except ImportError as e:
    print('Reinstalling packages...')
    subprocess.run(['pip', 'install', '-q', 'faiss-cpu', 'gensim'])

# GPU still attached
import torch
assert torch.cuda.is_available(), 'GPU not available — change runtime type!'
print('GPU OK:', torch.cuda.get_device_name(0))

print('\n✅ All systems go. You can run the tuning cells now.')

Drive OK: /content/drive/MyDrive/ICSRec_project
ICSRec OK
Data OK: /content/ICSRec/data/Grocery_and_Gourmet_Food.txt
Supervised data OK
Packages OK
GPU OK: NVIDIA A100-SXM4-40GB

✅ All systems go. You can run the tuning cells now.


# Run 2: intent_num=128

In [13]:
%cd /content/ICSRec/src
!python main.py \
    --data_name Grocery_and_Gourmet_Food \
    --rec_weight 1.0 \
    --lambda_0 0.3 \
    --beta_0 0.1 \
    --f_neg \
    --intent_num 128 \
    --hidden_dropout_prob 0.5 \
    --attention_probs_dropout_prob 0.5 \
    --model_idx 1 \
    --epochs 200

/content/ICSRec/src
Using Cuda: True
--------------------Configure Info:------------
data_dir                       :                            ../data/
output_dir                     :                              output
data_name                      :            Grocery_and_Gourmet_Food
encoder                        :                                 SAS
do_eval                        :                                   0
model_idx                      :                                   1
gpu_id                         :                                   0
noise_ratio                    :                                 0.0
temperature                    :                                 1.0
intent_num                     :                                 128
sim                            :                                 dot
model_name                     :                              ICSRec
hidden_size                    :                                  64
num_hidden_layers 

In [14]:
# Backup Run 2 (intent_num=128) and read its test result
import os, shutil
backup_dir = os.path.join(DRIVE_DIR, 'tuning_intent128')
os.makedirs(backup_dir, exist_ok=True)
for fn in os.listdir('/content/ICSRec/src/output'):
    if 'Grocery_and_Gourmet_Food-1' in fn:
        shutil.copy(f'/content/ICSRec/src/output/{fn}', f'{backup_dir}/{fn}')
        print(f'Backed up: {fn}')

log = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-1.txt'
if os.path.exists(log):
    print('\n===== Final result for intent_num=128 =====')
    with open(log) as f:
        for line in f.readlines()[-3:]:
            print(line.rstrip())

Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-1.pt
Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-1.txt

===== Final result for intent_num=128 =====
{'Epoch': 0, 'HIT@5': '0.0337', 'NDCG@5': '0.0215', 'HIT@10': '0.0531', 'NDCG@10': '0.0278', 'HIT@20': '0.0830', 'NDCG@20': '0.0352'}
ICSRec-SAS-Grocery_and_Gourmet_Food-1
{'Epoch': 0, 'HIT@5': '0.0337', 'NDCG@5': '0.0215', 'HIT@10': '0.0531', 'NDCG@10': '0.0278', 'HIT@20': '0.0830', 'NDCG@20': '0.0352'}


#  Run 3 (intent_num=512)

In [15]:
%cd /content/ICSRec/src
!python main.py \
    --data_name Grocery_and_Gourmet_Food \
    --rec_weight 1.0 \
    --lambda_0 0.3 \
    --beta_0 0.1 \
    --f_neg \
    --intent_num 512 \
    --hidden_dropout_prob 0.5 \
    --attention_probs_dropout_prob 0.5 \
    --model_idx 2 \
    --epochs 200

/content/ICSRec/src
Using Cuda: True
--------------------Configure Info:------------
data_dir                       :                            ../data/
output_dir                     :                              output
data_name                      :            Grocery_and_Gourmet_Food
encoder                        :                                 SAS
do_eval                        :                                   0
model_idx                      :                                   2
gpu_id                         :                                   0
noise_ratio                    :                                 0.0
temperature                    :                                 1.0
intent_num                     :                                 512
sim                            :                                 dot
model_name                     :                              ICSRec
hidden_size                    :                                  64
num_hidden_layers 

In [16]:
import os, shutil
backup_dir = os.path.join(DRIVE_DIR, 'tuning_intent512')
os.makedirs(backup_dir, exist_ok=True)
for fn in os.listdir('/content/ICSRec/src/output'):
    if 'Grocery_and_Gourmet_Food-2' in fn:
        shutil.copy(f'/content/ICSRec/src/output/{fn}', f'{backup_dir}/{fn}')
        print(f'Backed up: {fn}')

log = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-2.txt'
if os.path.exists(log):
    print('\n===== Final result for intent_num=512 =====')
    with open(log) as f:
        for line in f.readlines()[-3:]:
            print(line.rstrip())

Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-2.txt
Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-2.pt

===== Final result for intent_num=512 =====
{'Epoch': 0, 'HIT@5': '0.0334', 'NDCG@5': '0.0215', 'HIT@10': '0.0540', 'NDCG@10': '0.0281', 'HIT@20': '0.0842', 'NDCG@20': '0.0357'}
ICSRec-SAS-Grocery_and_Gourmet_Food-2
{'Epoch': 0, 'HIT@5': '0.0334', 'NDCG@5': '0.0215', 'HIT@10': '0.0540', 'NDCG@10': '0.0281', 'HIT@20': '0.0842', 'NDCG@20': '0.0357'}


# Run 4 (intent_num=64)

In [17]:
%cd /content/ICSRec/src
!python main.py \
    --data_name Grocery_and_Gourmet_Food \
    --rec_weight 1.0 \
    --lambda_0 0.3 \
    --beta_0 0.1 \
    --f_neg \
    --intent_num 64 \
    --hidden_dropout_prob 0.5 \
    --attention_probs_dropout_prob 0.5 \
    --model_idx 3 \
    --epochs 200

/content/ICSRec/src
Using Cuda: True
--------------------Configure Info:------------
data_dir                       :                            ../data/
output_dir                     :                              output
data_name                      :            Grocery_and_Gourmet_Food
encoder                        :                                 SAS
do_eval                        :                                   0
model_idx                      :                                   3
gpu_id                         :                                   0
noise_ratio                    :                                 0.0
temperature                    :                                 1.0
intent_num                     :                                  64
sim                            :                                 dot
model_name                     :                              ICSRec
hidden_size                    :                                  64
num_hidden_layers 

In [18]:
import os, shutil
backup_dir = os.path.join(DRIVE_DIR, 'tuning_intent64')
os.makedirs(backup_dir, exist_ok=True)
for fn in os.listdir('/content/ICSRec/src/output'):
    if 'Grocery_and_Gourmet_Food-3' in fn:
        shutil.copy(f'/content/ICSRec/src/output/{fn}', f'{backup_dir}/{fn}')
        print(f'Backed up: {fn}')

log = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-3.txt'
if os.path.exists(log):
    print('\n===== Final result for intent_num=64 =====')
    with open(log) as f:
        for line in f.readlines()[-3:]:
            print(line.rstrip())

Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-3.pt
Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-3.txt

===== Final result for intent_num=64 =====
{'Epoch': 0, 'HIT@5': '0.0328', 'NDCG@5': '0.0216', 'HIT@10': '0.0506', 'NDCG@10': '0.0274', 'HIT@20': '0.0827', 'NDCG@20': '0.0354'}
ICSRec-SAS-Grocery_and_Gourmet_Food-3
{'Epoch': 0, 'HIT@5': '0.0328', 'NDCG@5': '0.0216', 'HIT@10': '0.0506', 'NDCG@10': '0.0274', 'HIT@20': '0.0827', 'NDCG@20': '0.0354'}


# Final comparison table of all 4 runs

In [19]:
import os, re

print('=' * 80)
print('Hyperparameter tuning summary — final TEST metrics')
print('=' * 80)
print(f'{"intent_num":<12} {"HIT@5":<10} {"NDCG@5":<10} {"HIT@10":<10} {"NDCG@10":<10} {"HIT@20":<10} {"NDCG@20":<10}')
print('-' * 80)

for idx, intent_num in [(0, 256), (1, 128), (2, 512), (3, 64)]:
    log = f'/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-{idx}.txt'
    if not os.path.exists(log):
        print(f'{intent_num:<12} (log not found)')
        continue
    with open(log) as f:
        last_lines = f.readlines()[-10:]
    # Find the LAST line containing HIT@5 — that's the test result
    test_line = None
    for line in reversed(last_lines):
        if 'HIT@5' in line:
            test_line = line
            break
    if test_line:
        nums = re.findall(r"[\d.]+", test_line)
        # first number is epoch, then 6 metrics
        metrics = nums[1:7] if len(nums) >= 7 else nums
        print(f'{intent_num:<12} ' + ' '.join(f'{m:<10}' for m in metrics))
    else:
        print(f'{intent_num:<12} (could not parse)')

#  save this comparison to Drive
out = os.path.join(DRIVE_DIR, 'tuning_comparison.txt')
with open(out, 'w') as f:
    f.write('Hyperparameter tuning summary — final TEST metrics\n')
    f.write('=' * 80 + '\n')
    f.write(f'{"intent_num":<12} {"HIT@5":<10} {"NDCG@5":<10} {"HIT@10":<10} {"NDCG@10":<10} {"HIT@20":<10} {"NDCG@20":<10}\n')
    for idx, intent_num in [(0, 256), (1, 128), (2, 512), (3, 64)]:
        log = f'/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-{idx}.txt'
        if not os.path.exists(log):
            f.write(f'{intent_num:<12} (log not found)\n')
            continue
        with open(log) as lg:
            last_lines = lg.readlines()[-10:]
        test_line = None
        for line in reversed(last_lines):
            if 'HIT@5' in line:
                test_line = line
                break
        if test_line:
            nums = re.findall(r"[\d.]+", test_line)
            metrics = nums[1:7] if len(nums) >= 7 else nums
            f.write(f'{intent_num:<12} ' + ' '.join(f'{m:<10}' for m in metrics) + '\n')
print(f'\n✅ Comparison saved to {out}')

Hyperparameter tuning summary — final TEST metrics
intent_num   HIT@5      NDCG@5     HIT@10     NDCG@10    HIT@20     NDCG@20   
--------------------------------------------------------------------------------
256          5          0.0336     5          0.0217     10         0.0514    
128          5          0.0337     5          0.0215     10         0.0531    
512          5          0.0334     5          0.0215     10         0.0540    
64           5          0.0328     5          0.0216     10         0.0506    

✅ Comparison saved to /content/drive/MyDrive/ICSRec_project/tuning_comparison.txt
